# Part 3: Challenge

Forward(predict) + Inverse(find_optimal)를 활용하여 현장 시나리오를 풀어봅니다.

---

In [ ]:
#@title [준비] 실행 버튼(>)만 눌러주세요
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.font_manager as fm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.optimize import minimize

# 한글 폰트 설정 (Colab 환경)
for font in fm.findSystemFonts(fontpaths=None, fontext='ttf'):
    if 'Nanum' in font:
        fm.fontManager.addfont(font)
plt.rcParams['font.family'] = 'NanumBarunGothic'
plt.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def _gen(seed, x_cfg, y_funcs, n=1000):
    rng = np.random.default_rng(seed)
    xs = {nm: rng.uniform(lo, hi, n) for nm, (lo, hi) in x_cfg.items()}
    ys = {nm: fn(xs, rng) for nm, fn in y_funcs.items()}
    return pd.DataFrame({**xs, **ys})

df = _gen(42,
    {'온도_C': (50,200), '압력_kPa': (100,500), '유량_Lmin': (1,10), '교반_RPM': (100,800), '시간_min': (10,120)},
    {
        '순도_pct': lambda x, r: np.clip(80+0.05*x['온도_C']+0.01*x['압력_kPa']-0.3*x['유량_Lmin']+0.005*x['교반_RPM']+0.02*x['시간_min']-0.0001*x['온도_C']*x['압력_kPa']+0.0002*x['온도_C']*x['교반_RPM']+r.normal(0,1.5,1000), 85, 99.9),
        '입자크기_um': lambda x, r: np.clip(25-0.05*x['온도_C']+0.01*x['압력_kPa']+0.8*x['유량_Lmin']-0.01*x['교반_RPM']-0.03*x['시간_min']+0.0005*x['유량_Lmin']*x['교반_RPM']+r.normal(0,1.0,1000), 1, 50),
        '점도_cP': lambda x, r: np.clip(50+0.3*x['온도_C']+0.1*x['압력_kPa']-2.0*x['유량_Lmin']+0.05*x['교반_RPM']+0.5*x['시간_min']-0.001*x['온도_C']*x['유량_Lmin']+r.normal(0,5,1000), 30, 400),
    }
)

input_cols = ['온도_C', '압력_kPa', '유량_Lmin', '교반_RPM', '시간_min']
output_cols = ['순도_pct', '입자크기_um', '점도_cP']
specs = {'순도_pct': (95, 99.9), '입자크기_um': (5, 20), '점도_cP': (80, 250)}

X_tr, X_te, y_tr, y_te = train_test_split(df[input_cols], df[output_cols], test_size=0.2, random_state=42)
factory_model = RandomForestRegressor(n_estimators=100, random_state=42)
factory_model.fit(X_tr, y_tr)

def find_optimal(targets, n_restarts=30):
    bounds = [(50,200),(100,500),(1,10),(100,800),(10,120)]
    t_arr = np.array([targets[c] for c in output_cols])
    scales = np.array([specs[c][1]-specs[c][0] for c in output_cols])
    def obj(x):
        return np.sqrt(np.sum(((factory_model.predict([x])[0]-t_arr)/scales)**2))
    best = None
    for _ in range(n_restarts):
        x0 = [np.random.uniform(lo,hi) for lo,hi in bounds]
        res = minimize(obj, x0, bounds=bounds, method='L-BFGS-B')
        if best is None or res.fun < best.fun: best = res
    return dict(zip(input_cols, best.x)), dict(zip(output_cols, factory_model.predict([best.x])[0]))

print('준비 완료!')
print(f'df: 공장 데이터 {len(df)} 로트')
print(f'factory_model.predict([[온도, 압력, 유량, 교반, 시간]]): 품질 예측')
print(f'find_optimal({{순도_pct: 97, 입자크기_um: 10, 점도_cP: 150}}): 최적 조건 탐색')
print()
print('규격:')
for col, (lo, hi) in specs.items():
    print(f'  {col}: {lo} ~ {hi}')

---

## Mission 1: 납기 대응 -- 반응 시간 단축

**배경:** 화학 소재 회사 공정 엔지니어. 디스플레이용 코팅 원료 생산. 현재 1배치 50분.

**상황:** 다음 달 물량 30% 증가. 설비 추가 불가. **반응 시간을 35분으로 줄여야** 함.

**문제:** 시간 줄이면 순도, 입자크기, 점도가 규격 이탈 가능.

### TODO

1. `factory_model.predict(...)` 에서 **시간을 35로 고정**하고, 나머지 조건을 조절해서 **전 항목 규격 이내**를 찾으세요.
2. 찾으셨으면 공유 시트 "챌린지" 칸에 조건을 적어주세요.

| 품질 | 규격 |
|------|------|
| 순도 | 95 ~ 99.9% |
| 입자크기 | 5 ~ 20 um |
| 점도 | 80 ~ 250 cP |

**힌트:** 변수를 한꺼번에 다 바꾸지 말고, 하나씩 움직여 보세요.

**심화:** 찾으셨으면 `find_optimal`로 같은 문제를 풀어서 수동 결과와 비교해 보세요.

In [ ]:
# 여기서 시작하세요
# 예: factory_model.predict([[150, 300, 5, 400, 35]])

---

## Mission 2: 긴급 주문

**상황:** 영업팀 긴급 요청. 고객 스펙:

| 품질 | 목표 |
|------|------|
| 순도 | 97.5% 이상 |
| 입자크기 | 9.0 ~ 11.0 um |
| 점도 | 140 ~ 160 cP |

**제약:** 온도 190도 이하 (설비 점검 중), 반응시간 50분 이내

### TODO

1. `find_optimal`로 목표 품질 입력 -> 최적 조건 탐색
2. 온도가 190 넘으면 -> `factory_model.predict`에서 온도 190으로 고정, 나머지 조절
3. Colab AI에게: **"온도 190 이하인 로트 중 순도가 가장 높은 5건을 보여줘"**
4. 과거 실적과 AI 예측을 비교

In [ ]:
# 여기서 시작하세요
# 예: find_optimal({'순도_pct': 97.5, '입자크기_um': 10, '점도_cP': 150})

---

## Mission 3: 자유 분석 (보너스)

시간이 남으면 Colab AI에게 자유롭게 질문해 보세요.

**프롬프트 예시:**
- "순도가 97% 이상인 로트들의 공통 조건을 분석해줘"
- "온도와 압력의 상호작용 효과를 3D 그래프로 그려줘"
- "변수 중요도를 막대 그래프로 보여줘"
- "규격 이탈 로트의 특성을 정리해줘"

In [ ]:
# 자유롭게 시도해 보세요